# 🚬 흡연 분류 V12 - AutoML + OOF Strict

## 핵심 원칙
- ✅ **모든 선택은 OOF(CV)로만** - 추측/예상 금지
- ✅ **Baseline vs AutoML OOF 비교** - 더 높은 것만 채택
- ✅ **임계값도 OOF로만 최적화** - train in-sample 금지
- ✅ **안전장치 필수** - 파일 체크, 상수 피처 제거, dtype 검증

---

## 🔧 CONFIG (상단 토글)

In [ ]:
#===========================================
# 상단 토글 변수 (사용자 설정)
#===========================================
SEED = 42

# AutoML 설정
USE_AUTOML = True
AUTOML_ENGINE = "autogluon"  # "autogluon" or "flaml"
AUTOML_BUDGET = 600  # 초 단위 (None이면 AutoML OFF)

# 전처리 옵션
MISSING_METHOD = "A2"  # A1: fillna(0), A2: median/mode, A3: A2+flag
OUTLIER_METHOD = "B2"  # B1: OFF, B2: quantile clip
OUTLIER_QUANTILE = (0.01, 0.99)  # clip 범위

# 피처 그룹 ON/OFF
USE_G1_RATIO = True      # TG/HDL, HDL/LDL 등
USE_G2_LOG = True        # log1p, 제곱
USE_G3_HIGH_FLAG = True  # quantile 기반 high 플래그
USE_G4_INTERACTION = True  # hemo×gtp 등
USE_G5_BINNING = True    # age_group, bmi_group

# high 플래그 분위수
HIGH_QUANTILE = 0.80

print("✅ CONFIG 설정 완료")

## STEP 0: 환경 설정 + 안전장치

In [ ]:
# 기본 라이브러리
!pip install -q xgboost lightgbm catboost

In [ ]:
# AutoML 설치 (조건부)
AUTOML_AVAILABLE = False
AUTOML_ENGINE_USED = None

if USE_AUTOML and AUTOML_BUDGET is not None:
    if AUTOML_ENGINE == "autogluon":
        try:
            !pip install -q autogluon.tabular
            from autogluon.tabular import TabularPredictor
            AUTOML_AVAILABLE = True
            AUTOML_ENGINE_USED = "autogluon"
            print("✅ AutoGluon 설치 성공")
        except Exception as e:
            print(f"⚠️ AutoGluon 설치 실패: {e}")
            print("   → FLAML로 fallback 시도")
            try:
                !pip install -q flaml
                from flaml import AutoML
                AUTOML_AVAILABLE = True
                AUTOML_ENGINE_USED = "flaml"
                print("✅ FLAML 설치 성공 (fallback)")
            except:
                print("❌ FLAML도 설치 실패 → AutoML OFF")
    elif AUTOML_ENGINE == "flaml":
        try:
            !pip install -q flaml
            from flaml import AutoML
            AUTOML_AVAILABLE = True
            AUTOML_ENGINE_USED = "flaml"
            print("✅ FLAML 설치 성공")
        except Exception as e:
            print(f"❌ FLAML 설치 실패: {e} → AutoML OFF")
else:
    print("ℹ️ AutoML 비활성화 (USE_AUTOML=False 또는 AUTOML_BUDGET=None)")

print(f"\n📊 AutoML 상태: {'ON (' + AUTOML_ENGINE_USED + ')' if AUTOML_AVAILABLE else 'OFF'}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
set_seed(SEED)

# 경로 설정 (고정)
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

# result_path 폴더 생성
os.makedirs(result_path, exist_ok=True)
print(f"✅ result_path 생성/확인: {result_path}")

In [ ]:
#===========================================
# 안전장치 1: 파일 존재 여부 체크
#===========================================
print("=" * 60)
print("🔍 안전장치 1: 파일 존재 여부 체크")
print("=" * 60)

files_to_check = {
    'train.csv': train_path,
    'test.csv': test_path,
    'sample_submission.csv': submission_path
}

all_exist = True
for name, path in files_to_check.items():
    exists = os.path.exists(path)
    status = "OK" if exists else "NOT FOUND"
    print(f"   {name}: {status}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("❌ 필수 파일이 없습니다. 경로를 확인하세요.")

print("\n✅ 모든 파일 존재 확인 완료!")

## STEP 1: 데이터 로드 + 검증

In [ ]:
train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("=" * 60)
print("📊 STEP 1: 데이터 로드 완료")
print("=" * 60)
print(f"Train shape: {train_raw.shape}")
print(f"Test shape: {test_raw.shape}")
print(f"Submission shape: {submission.shape}")

In [ ]:
#===========================================
# 안전장치 2: 컬럼명, 중복 컬럼, 결측치
#===========================================
print("\n" + "=" * 60)
print("🔍 안전장치 2: 컬럼 및 결측치 검증")
print("=" * 60)

print(f"\n📋 Train 컬럼명 ({len(train_raw.columns)}개):")
print(train_raw.columns.tolist())

print(f"\n📋 Test 컬럼명 ({len(test_raw.columns)}개):")
print(test_raw.columns.tolist())

# 중복 컬럼 체크
train_dup = train_raw.columns[train_raw.columns.duplicated()].tolist()
test_dup = test_raw.columns[test_raw.columns.duplicated()].tolist()
print(f"\n🔍 중복 컬럼:")
print(f"   Train: {train_dup if train_dup else '없음'}")
print(f"   Test: {test_dup if test_dup else '없음'}")

# 결측치 비율 상위 20개
print(f"\n🔍 결측치 비율 (상위 20개):")
missing_train = (train_raw.isnull().sum() / len(train_raw) * 100).sort_values(ascending=False)
missing_test = (test_raw.isnull().sum() / len(test_raw) * 100).sort_values(ascending=False)

print("\nTrain:")
print(missing_train.head(20).to_string())
print("\nTest:")
print(missing_test.head(20).to_string())

## STEP 2: 컬럼 매핑

In [ ]:
# 컬럼 매핑 (한글 → 영어)
COL_MAP = {
    'ID': 'id',
    '나이': 'age',
    '키(cm)': 'height_cm',
    '몸무게(kg)': 'weight_kg',
    'BMI': 'bmi',
    '시력(좌)': 'eyesight_left',
    '시력(우)': 'eyesight_right',
    '청력(좌)': 'hearing_left',
    '청력(우)': 'hearing_right',
    '충치': 'cavity',
    '공복 혈당': 'fasting_glucose',
    '수축기 혈압': 'systolic_bp',
    '이완기 혈압': 'diastolic_bp',
    '혈압': 'blood_pressure',
    '중성 지방': 'triglyceride',
    '혈청 크레아티닌': 'serum_creatinine',
    '콜레스테롤': 'cholesterol',
    '고밀도 지단백': 'hdl',
    '고밀도지단백': 'hdl',
    '저밀도 지단백': 'ldl',
    '저밀도지단백': 'ldl',
    '헤모글로빈': 'hemoglobin',
    '요 단백': 'urine_protein',
    '간 효소율': 'gtp',
    'label': 'label'
}

def rename_columns(df):
    """컬럼명 매핑 (존재하는 것만)"""
    rename_dict = {}
    for old, new in COL_MAP.items():
        if old in df.columns:
            rename_dict[old] = new
    return df.rename(columns=rename_dict)

train = rename_columns(train_raw.copy())
test = rename_columns(test_raw.copy())

print("✅ 컬럼 매핑 완료")
print(f"매핑 후 Train 컬럼: {train.columns.tolist()}")

In [ ]:
# ID 분리
if 'id' in test.columns:
    test_ids = test['id'].copy()
    train = train.drop('id', axis=1, errors='ignore')
    test = test.drop('id', axis=1, errors='ignore')
else:
    test_ids = pd.Series(range(len(test)))

# X, y 분리
y = train['label'].copy()
X = train.drop('label', axis=1)
X_test = test.drop('label', axis=1, errors='ignore')

print(f"\nX: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")
print(f"클래스 분포: {y.value_counts().to_dict()}")

## STEP 3: 전처리 함수 정의

In [ ]:
def preprocess_missing(X_train, X_val, X_test_data, method):
    """결측치 처리"""
    X_tr = X_train.copy()
    X_va = X_val.copy()
    X_te = X_test_data.copy()
    
    if method == "A1":  # fillna(0)
        X_tr = X_tr.fillna(0)
        X_va = X_va.fillna(0)
        X_te = X_te.fillna(0)
        
    elif method in ["A2", "A3"]:  # median/mode
        for col in X_tr.columns:
            if X_tr[col].dtype in ['float64', 'int64', 'float32', 'int32']:
                fill_val = X_tr[col].median()
            else:
                fill_val = X_tr[col].mode().iloc[0] if len(X_tr[col].mode()) > 0 else 0
            X_tr[col] = X_tr[col].fillna(fill_val)
            X_va[col] = X_va[col].fillna(fill_val)
            X_te[col] = X_te[col].fillna(fill_val)
            
        if method == "A3":  # 결측 플래그 추가
            for col in X_train.columns:
                if X_train[col].isnull().sum() > 0:
                    flag_col = f"{col}_isna"
                    X_tr[flag_col] = X_train[col].isnull().astype(int)
                    X_va[flag_col] = X_val[col].isnull().astype(int)
                    X_te[flag_col] = X_test_data[col].isnull().astype(int)
    
    return X_tr, X_va, X_te


def preprocess_outliers(X_train, X_val, X_test_data, method, quantiles):
    """이상치 처리"""
    X_tr = X_train.copy()
    X_va = X_val.copy()
    X_te = X_test_data.copy()
    
    if method == "B2":  # quantile clip
        q_low, q_high = quantiles
        for col in X_tr.select_dtypes(include=[np.number]).columns:
            lower = X_tr[col].quantile(q_low)
            upper = X_tr[col].quantile(q_high)
            X_tr[col] = X_tr[col].clip(lower, upper)
            X_va[col] = X_va[col].clip(lower, upper)
            X_te[col] = X_te[col].clip(lower, upper)
    
    return X_tr, X_va, X_te

print("✅ 전처리 함수 정의 완료")

## STEP 4: 피처 엔지니어링 함수 정의

In [ ]:
def create_features(df, df_train_for_quantile=None, high_q=0.80, verbose=False):
    """
    피처 엔지니어링 (그룹별 ON/OFF)
    df_train_for_quantile: quantile 계산용 train 데이터 (val/test 처리 시)
    """
    df = df.copy()
    cols = df.columns.tolist()
    ref = df_train_for_quantile if df_train_for_quantile is not None else df
    
    created_features = []
    
    # G1: 비율 피처
    if USE_G1_RATIO:
        if 'triglyceride' in cols and 'hdl' in cols:
            df['tg_hdl_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
            created_features.append('tg_hdl_ratio')
        if 'hdl' in cols and 'ldl' in cols:
            df['hdl_ldl_ratio'] = df['hdl'] / (df['ldl'] + 1)
            created_features.append('hdl_ldl_ratio')
        if 'triglyceride' in cols and 'hdl' in cols and 'ldl' in cols:
            df['tg_hdl_ldl_ratio'] = df['triglyceride'] / (df['hdl'] + df['ldl'] + 1)
            created_features.append('tg_hdl_ldl_ratio')
    
    # G2: 로그/제곱
    if USE_G2_LOG:
        if 'gtp' in cols:
            df['gtp_log'] = np.log1p(df['gtp'])
            created_features.append('gtp_log')
        if 'triglyceride' in cols:
            df['tg_log'] = np.log1p(df['triglyceride'])
            created_features.append('tg_log')
        if 'hemoglobin' in cols:
            df['hemo_sq'] = df['hemoglobin'] ** 2
            created_features.append('hemo_sq')
    
    # G3: high 플래그 (quantile 기반)
    if USE_G3_HIGH_FLAG:
        for col in ['hemoglobin', 'gtp', 'triglyceride']:
            if col in cols:
                threshold = ref[col].quantile(high_q)
                flag_col = f"{col}_high"
                df[flag_col] = (df[col] > threshold).astype(int)
                created_features.append(flag_col)
                if verbose:
                    print(f"   {flag_col}: threshold={threshold:.4f} (q={high_q})")
    
    # G4: 상호작용
    if USE_G4_INTERACTION:
        if 'hemoglobin' in cols and 'gtp' in cols:
            df['hemo_x_gtp'] = df['hemoglobin'] * df['gtp']
            created_features.append('hemo_x_gtp')
        if 'age' in cols and 'hemoglobin' in cols:
            df['age_x_hemo'] = df['age'] * df['hemoglobin']
            created_features.append('age_x_hemo')
        if 'age' in cols and 'gtp' in cols:
            df['age_x_gtp'] = df['age'] * df['gtp']
            created_features.append('age_x_gtp')
    
    # G5: 구간화
    if USE_G5_BINNING:
        if 'age' in cols:
            df['age_group'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100], 
                                     labels=['0', '1', '2', '3', '4']).astype(str)
            created_features.append('age_group')
        if 'bmi' in cols:
            df['bmi_group'] = pd.cut(df['bmi'], bins=[0, 18.5, 23, 25, 30, 100], 
                                     labels=['0', '1', '2', '3', '4']).astype(str)
            created_features.append('bmi_group')
    
    # 결측치/무한값 처리
    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)
    
    return df, created_features

print("✅ 피처 엔지니어링 함수 정의 완료")

In [ ]:
#===========================================
# 안전장치 3: 파생피처 상수 여부 체크
#===========================================
print("\n" + "=" * 60)
print("🔍 안전장치 3: 파생피처 상수 여부 체크")
print("=" * 60)

# 테스트용으로 전체 데이터에 피처 생성
X_test_fe, created = create_features(X.copy(), high_q=HIGH_QUANTILE, verbose=True)

print(f"\n📋 생성된 피처 ({len(created)}개): {created}")

# 상수 피처 체크
const_features = []
for col in created:
    if col in X_test_fe.columns:
        vc = X_test_fe[col].value_counts()
        if len(vc) == 1:
            const_features.append(col)
            print(f"   ⚠️ {col}: 상수 피처 (값={vc.index[0]})")
        else:
            print(f"   ✅ {col}: {len(vc)} unique values")

if const_features:
    print(f"\n⚠️ 상수 피처 {len(const_features)}개 발견 → 자동 제외: {const_features}")

## STEP 5: CV 설정 (공통)

In [ ]:
# 공통 CV 설정
N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print(f"✅ CV 설정: StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={SEED})")

## STEP 6: BASELINE OOF 생성

In [ ]:
print("\n" + "=" * 60)
print("🔄 STEP 6: BASELINE OOF 생성")
print("=" * 60)

# OOF 저장
oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

# Test 예측 저장
test_xgb = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

# CatBoost 범주형 컬럼 후보
cat_feature_candidates = ['cavity', 'urine_protein', 'age_group', 'bmi_group']

fold_scores = {'xgb': [], 'lgb': [], 'cat': []}

In [ ]:
for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    
    X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    X_te = X_test.copy()
    
    # 전처리: 결측치
    X_tr, X_va, X_te = preprocess_missing(X_tr, X_va, X_te, MISSING_METHOD)
    
    # 전처리: 이상치
    X_tr, X_va, X_te = preprocess_outliers(X_tr, X_va, X_te, OUTLIER_METHOD, OUTLIER_QUANTILE)
    
    # 피처 엔지니어링
    X_tr, _ = create_features(X_tr, df_train_for_quantile=X_tr, high_q=HIGH_QUANTILE)
    X_va, _ = create_features(X_va, df_train_for_quantile=X_tr, high_q=HIGH_QUANTILE)
    X_te, _ = create_features(X_te, df_train_for_quantile=X_tr, high_q=HIGH_QUANTILE)
    
    # 상수 피처 제거
    if const_features:
        X_tr = X_tr.drop(columns=const_features, errors='ignore')
        X_va = X_va.drop(columns=const_features, errors='ignore')
        X_te = X_te.drop(columns=const_features, errors='ignore')
    
    # 컬럼 순서 맞추기
    common_cols = [c for c in X_tr.columns if c in X_te.columns]
    X_tr = X_tr[common_cols]
    X_va = X_va[common_cols]
    X_te = X_te[common_cols]
    
    #===========================================
    # 안전장치 4: CatBoost cat_features dtype 변환
    #===========================================
    cat_features_actual = [c for c in cat_feature_candidates if c in X_tr.columns]
    for col in cat_features_actual:
        X_tr[col] = X_tr[col].astype(str)
        X_va[col] = X_va[col].astype(str)
        X_te[col] = X_te[col].astype(str)
    
    # XGBoost, LightGBM용 (수치형만)
    X_tr_num = X_tr.copy()
    X_va_num = X_va.copy()
    X_te_num = X_te.copy()
    for col in cat_features_actual:
        X_tr_num[col] = pd.factorize(X_tr_num[col])[0]
        X_va_num[col] = pd.factorize(X_va_num[col])[0]
        X_te_num[col] = pd.factorize(X_te_num[col])[0]
    
    # XGBoost
    xgb_model = XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss'
    )
    xgb_model.fit(X_tr_num, y_tr)
    oof_xgb[va_idx] = xgb_model.predict_proba(X_va_num)[:, 1]
    test_xgb += xgb_model.predict_proba(X_te_num)[:, 1] / N_SPLITS
    xgb_acc = accuracy_score(y_va, (oof_xgb[va_idx] >= 0.5).astype(int))
    fold_scores['xgb'].append(xgb_acc)
    
    # LightGBM
    lgb_model = LGBMClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=SEED, verbose=-1
    )
    lgb_model.fit(X_tr_num, y_tr)
    oof_lgb[va_idx] = lgb_model.predict_proba(X_va_num)[:, 1]
    test_lgb += lgb_model.predict_proba(X_te_num)[:, 1] / N_SPLITS
    lgb_acc = accuracy_score(y_va, (oof_lgb[va_idx] >= 0.5).astype(int))
    fold_scores['lgb'].append(lgb_acc)
    
    # CatBoost
    cat_model = CatBoostClassifier(
        iterations=500, depth=5, learning_rate=0.03,
        auto_class_weights='Balanced', random_state=SEED, verbose=0,
        cat_features=cat_features_actual if cat_features_actual else None
    )
    cat_model.fit(X_tr, y_tr)
    oof_cat[va_idx] = cat_model.predict_proba(X_va)[:, 1]
    test_cat += cat_model.predict_proba(X_te)[:, 1] / N_SPLITS
    cat_acc = accuracy_score(y_va, (oof_cat[va_idx] >= 0.5).astype(int))
    fold_scores['cat'].append(cat_acc)
    
    print(f"   XGB: {xgb_acc:.5f}, LGB: {lgb_acc:.5f}, CAT: {cat_acc:.5f}")

print("\n✅ Baseline OOF 생성 완료!")

In [ ]:
# 모델별 OOF 정확도
print("\n" + "=" * 60)
print("📊 Baseline 모델별 OOF Accuracy")
print("=" * 60)

baseline_results = {}

for name, oof in [('XGBoost', oof_xgb), ('LightGBM', oof_lgb), ('CatBoost', oof_cat)]:
    acc = accuracy_score(y, (oof >= 0.5).astype(int))
    baseline_results[name] = acc
    print(f"{name:12}: OOF Accuracy = {acc:.5f}")

# 앙상블: 단순 평균
oof_avg = (oof_xgb + oof_lgb + oof_cat) / 3
acc_avg = accuracy_score(y, (oof_avg >= 0.5).astype(int))
baseline_results['Ensemble_Avg'] = acc_avg
print(f"{'Ensemble_Avg':12}: OOF Accuracy = {acc_avg:.5f}")

# 앙상블: 가중 평균 후보 (4~6개만)
weight_candidates = [
    (0.4, 0.35, 0.25),
    (0.35, 0.35, 0.30),
    (0.4, 0.3, 0.3),
    (0.5, 0.3, 0.2),
    (0.33, 0.33, 0.34),
]

best_weights = None
best_weight_acc = 0

print("\n가중 평균 후보 비교:")
for w in weight_candidates:
    oof_w = w[0]*oof_xgb + w[1]*oof_lgb + w[2]*oof_cat
    acc_w = accuracy_score(y, (oof_w >= 0.5).astype(int))
    print(f"   {w}: {acc_w:.5f}")
    if acc_w > best_weight_acc:
        best_weight_acc = acc_w
        best_weights = w

print(f"\n🏆 최적 가중치: {best_weights} → OOF Accuracy = {best_weight_acc:.5f}")
baseline_results['Ensemble_Weighted'] = best_weight_acc

# 최종 Baseline OOF
oof_baseline = best_weights[0]*oof_xgb + best_weights[1]*oof_lgb + best_weights[2]*oof_cat
test_baseline = best_weights[0]*test_xgb + best_weights[1]*test_lgb + best_weights[2]*test_cat

BASELINE_OOF_ACC = best_weight_acc
print(f"\n✅ Baseline 최종 OOF Accuracy: {BASELINE_OOF_ACC:.5f}")

## STEP 7: AutoML OOF 생성 (옵션)

In [ ]:
print("\n" + "=" * 60)
print("🚀 STEP 7: AutoML OOF 생성")
print("=" * 60)

oof_automl = np.zeros(len(X))
test_automl = np.zeros(len(X_test))
AUTOML_OOF_ACC = 0

if not AUTOML_AVAILABLE:
    print("⏩ AutoML 비활성화 → SKIP")
else:
    print(f"\n🔄 AutoML 엔진: {AUTOML_ENGINE_USED}")
    print(f"   Budget: {AUTOML_BUDGET}초/fold")
    
    automl_fold_scores = []
    
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        print(f"\n--- AutoML Fold {fold+1}/{N_SPLITS} ---")
        
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_te = X_test.copy()
        
        # 전처리
        X_tr, X_va, X_te = preprocess_missing(X_tr, X_va, X_te, MISSING_METHOD)
        X_tr, X_va, X_te = preprocess_outliers(X_tr, X_va, X_te, OUTLIER_METHOD, OUTLIER_QUANTILE)
        X_tr, _ = create_features(X_tr, df_train_for_quantile=X_tr, high_q=HIGH_QUANTILE)
        X_va, _ = create_features(X_va, df_train_for_quantile=X_tr, high_q=HIGH_QUANTILE)
        X_te, _ = create_features(X_te, df_train_for_quantile=X_tr, high_q=HIGH_QUANTILE)
        
        if const_features:
            X_tr = X_tr.drop(columns=const_features, errors='ignore')
            X_va = X_va.drop(columns=const_features, errors='ignore')
            X_te = X_te.drop(columns=const_features, errors='ignore')
        
        common_cols = [c for c in X_tr.columns if c in X_te.columns]
        X_tr = X_tr[common_cols]
        X_va = X_va[common_cols]
        X_te = X_te[common_cols]
        
        if AUTOML_ENGINE_USED == "autogluon":
            # AutoGluon
            train_df = X_tr.copy()
            train_df['label'] = y_tr.values
            
            predictor = TabularPredictor(
                label='label', problem_type='binary', eval_metric='accuracy',
                path=f'/tmp/autogluon_fold{fold}', verbosity=0
            )
            predictor.fit(
                train_df, time_limit=AUTOML_BUDGET,
                presets='medium_quality', num_bag_folds=0,
                excluded_model_types=['KNN', 'NN_TORCH']
            )
            
            # 리더보드 출력
            lb = predictor.leaderboard(silent=True)
            if len(lb) > 0:
                best_model = lb.sort_values('score_val', ascending=False).iloc[0]['model']
                print(f"   Best model: {best_model}")
            
            # OOF 예측
            va_proba = predictor.predict_proba(X_va)
            if isinstance(va_proba, pd.DataFrame):
                oof_automl[va_idx] = va_proba[1].values if 1 in va_proba.columns else va_proba.iloc[:, -1].values
            else:
                oof_automl[va_idx] = va_proba[:, 1]
            
            # Test 예측
            te_proba = predictor.predict_proba(X_te)
            if isinstance(te_proba, pd.DataFrame):
                test_automl += (te_proba[1].values if 1 in te_proba.columns else te_proba.iloc[:, -1].values) / N_SPLITS
            else:
                test_automl += te_proba[:, 1] / N_SPLITS
                
        elif AUTOML_ENGINE_USED == "flaml":
            # FLAML
            automl = AutoML()
            automl.fit(X_tr, y_tr, task='classification', metric='accuracy', 
                      time_budget=AUTOML_BUDGET, verbose=0)
            
            print(f"   Best model: {automl.best_estimator}")
            
            oof_automl[va_idx] = automl.predict_proba(X_va)[:, 1]
            test_automl += automl.predict_proba(X_te)[:, 1] / N_SPLITS
        
        fold_acc = accuracy_score(y_va, (oof_automl[va_idx] >= 0.5).astype(int))
        automl_fold_scores.append(fold_acc)
        print(f"   Fold Accuracy: {fold_acc:.5f}")
    
    AUTOML_OOF_ACC = accuracy_score(y, (oof_automl >= 0.5).astype(int))
    print(f"\n✅ AutoML 최종 OOF Accuracy: {AUTOML_OOF_ACC:.5f}")

## STEP 8: Baseline vs AutoML 비교 → 채택

In [ ]:
print("\n" + "=" * 60)
print("📊 STEP 8: Baseline vs AutoML 비교")
print("=" * 60)

# OOF 비교표
comparison = pd.DataFrame({
    'Method': ['Baseline (Weighted Ensemble)', 'AutoML'],
    'OOF_Accuracy': [BASELINE_OOF_ACC, AUTOML_OOF_ACC]
})
print(comparison.to_string(index=False))

# 채택 결정
if AUTOML_OOF_ACC > BASELINE_OOF_ACC:
    SELECTED_METHOD = 'automl'
    SELECTED_OOF = oof_automl
    SELECTED_TEST = test_automl
    SELECTED_ACC = AUTOML_OOF_ACC
    print(f"\n🏆 AutoML 채택 (OOF: {AUTOML_OOF_ACC:.5f} > {BASELINE_OOF_ACC:.5f})")
else:
    SELECTED_METHOD = 'baseline'
    SELECTED_OOF = oof_baseline
    SELECTED_TEST = test_baseline
    SELECTED_ACC = BASELINE_OOF_ACC
    print(f"\n🏆 Baseline 채택 (OOF: {BASELINE_OOF_ACC:.5f} >= {AUTOML_OOF_ACC:.5f})")

# OOF 비교표 저장
comparison.to_csv(result_path + 'oof_summary_v12.csv', index=False)
print(f"\n✅ oof_summary_v12.csv 저장 완료")

## STEP 9: OOF 기반 임계값 최적화

In [ ]:
print("\n" + "=" * 60)
print("🔍 STEP 9: OOF 기반 임계값 최적화")
print("=" * 60)

# 임계값 sweep (0.30~0.70, step=0.01)
thresholds = np.arange(0.30, 0.71, 0.01)
results = []

for th in thresholds:
    pred = (SELECTED_OOF >= th).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    results.append({'threshold': round(th, 2), 'accuracy': acc, 'f1': f1})

results_df = pd.DataFrame(results)
print("\nThreshold별 OOF Accuracy (상위 15개):")
print(results_df.nlargest(15, 'accuracy').to_string(index=False))

# 최적 임계값
best_row = results_df.loc[results_df['accuracy'].idxmax()]
BEST_THRESHOLD = best_row['threshold']
BEST_OOF_ACC = best_row['accuracy']
BEST_F1 = best_row['f1']

print(f"\n🏆 최적 임계값: {BEST_THRESHOLD:.2f}")
print(f"   OOF Accuracy: {BEST_OOF_ACC:.5f}")
print(f"   OOF F1-Score: {BEST_F1:.5f}")

## STEP 10: 제출 파일 생성

In [ ]:
print("\n" + "=" * 60)
print("📁 STEP 10: 제출 파일 생성")
print("=" * 60)

# 제출 임계값들
submit_thresholds = [
    round(BEST_THRESHOLD - 0.02, 2),
    round(BEST_THRESHOLD, 2),
    round(BEST_THRESHOLD + 0.02, 2)
]

saved_files = []

for th in submit_thresholds:
    pred = (SELECTED_TEST >= th).astype(int)
    
    # OOF 정확도 (참고용)
    oof_pred = (SELECTED_OOF >= th).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    # 파일 생성
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    # 파일명
    th_str = str(int(th * 100)).zfill(2)
    if th == BEST_THRESHOLD:
        filename = f'submission_v12_{SELECTED_METHOD}_t{th_str}_best.csv'
    else:
        filename = f'submission_v12_{SELECTED_METHOD}_t{th_str}_pm.csv'
    
    filepath = result_path + filename
    sub.to_csv(filepath, index=False)
    saved_files.append(filename)
    
    n_pos = (pred == 1).sum()
    pct = n_pos / len(pred) * 100
    
    marker = "⭐" if th == BEST_THRESHOLD else "  "
    print(f"\n{marker} {filename}")
    print(f"   Threshold: {th:.2f}")
    print(f"   OOF Accuracy: {oof_acc:.5f}")
    print(f"   예측: 비흡연={len(pred)-n_pos} ({100-pct:.1f}%), 흡연={n_pos} ({pct:.1f}%)")

print(f"\n✅ {len(saved_files)}개 제출 파일 생성 완료!")

In [ ]:
# 저장된 파일 목록
print("\n📁 저장된 파일 목록:")
for f in saved_files:
    print(f"   ✅ {result_path}{f}")

print(f"   ✅ {result_path}oof_summary_v12.csv")

## STEP 11: 최종 요약

In [ ]:
print("\n" + "=" * 60)
print("🎉 V12 완료 - 최종 요약")
print("=" * 60)

print(f"\n📊 채택된 방법: {SELECTED_METHOD.upper()}")
print(f"   OOF Accuracy: {SELECTED_ACC:.5f}")
print(f"   Best Threshold: {BEST_THRESHOLD:.2f}")
print(f"   Best OOF Accuracy (with threshold): {BEST_OOF_ACC:.5f}")

print(f"\n🏆 최종 결과: method={SELECTED_METHOD} / OOF={BEST_OOF_ACC:.5f} / best_t={BEST_THRESHOLD:.2f}")

In [ ]:
# 다운로드
from google.colab import files

best_file = result_path + f'submission_v12_{SELECTED_METHOD}_t{str(int(BEST_THRESHOLD*100)).zfill(2)}_best.csv'
files.download(best_file)

print(f"\n📥 다운로드: {best_file.split('/')[-1]}")

In [ ]:
# 추가 파일 다운로드
print("\n📥 추가 파일 다운로드:")
for f in saved_files:
    if 'best' not in f:
        files.download(result_path + f)
        print(f"   ✅ {f}")